<a href="https://colab.research.google.com/github/nadayali710-spec/Programming-For-AI-Project/blob/main/Hockey_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hockey Team Performance Analysis and Prediction

In [1]:
"""Shared helpers for the Hockey Team Performance project.

Contains: paths, chronological split, custom transformers, a leakage-safe
preprocessing pipeline builder, and metric helpers.
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TEAM_ROLL_NUMBERS = ["25L-2558"]
SEED = 2558
ROOT = Path(os.environ.get("HOCKEY_PROJECT_DIR", Path.cwd()))
DATA, FIG, RES = ROOT / "data", ROOT / "figures", ROOT / "results"
for _p in (DATA, FIG, RES):
    _p.mkdir(parents=True, exist_ok=True)

RAW_CSV = DATA / "hockey_raw.csv"
RAW_BACKUP = DATA / "hockey_raw_UNCHANGED_COPY.csv"
RAW_META = DATA / "hockey_raw_metadata.json"
WORKING_CSV = DATA / "hockey_working_5pct_missing.csv"
CLEAN_CSV = DATA / "hockey_clean_eda.csv"

CAT = ["Team Name"]
NUM = ["Year", "GF", "GA", "+/-"]
FEATURES = CAT + NUM
def load_working():
    """Load the working copy (with controlled missing values), drop exact duplicates."""
    df = pd.read_csv(WORKING_CSV)
    data_cols = [c for c in df.columns if c != "collected_at"]
    return df.drop_duplicates(subset=data_cols).reset_index(drop=True)


def chronological_split(df, test_frac=0.2):
    """Train on earlier seasons, test on later ones.

    The cut is placed at a season boundary so that the training share is as
    close as possible to 1 - test_frac (whole seasons are never split).
    Returns (train, test, last_train_year).
    """
    df = df.sort_values("Year", kind="stable").reset_index(drop=True)
    share = df.groupby("Year").size().cumsum() / len(df)
    cutoff = (share - (1 - test_frac)).abs().idxmin()
    return df[df["Year"] <= cutoff].copy(), df[df["Year"] > cutoff].copy(), int(cutoff)


def skewed_columns(train, cols, threshold=0.5):
    """Columns suited to log1p: meaningfully RIGHT-skewed (skew > threshold) and
    non-negative. log1p compresses a long right tail; it would make a left-skewed
    column worse. Decided on TRAINING data only."""
    out = []
    for c in cols:
        s = train[c].dropna()
        if s.skew() > threshold and s.min() >= 0:
            out.append(c)
    return out


class PythagoreanExpectation(BaseEstimator, TransformerMixin):
    """Hockey Pythagorean expectation: GF^2 / (GF^2 + GA^2)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        gf2, ga2 = X["GF"] ** 2, X["GA"] ** 2
        denom = (gf2 + ga2).replace(0, np.nan)
        X["Pythagorean"] = (gf2 / denom).fillna(0.5)
        return X


class IQRCapper(BaseEstimator, TransformerMixin):
    """Clip columns to [Q1 - k*IQR, Q3 + k*IQR]; bounds learned on train."""

    def __init__(self, cols, k=1.5):
        self.cols, self.k = cols, k

    def fit(self, X, y=None):
        q1, q3 = X[self.cols].quantile(0.25), X[self.cols].quantile(0.75)
        iqr = q3 - q1
        self.lower_, self.upper_ = q1 - self.k * iqr, q3 + self.k * iqr
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = X[self.cols].clip(self.lower_, self.upper_, axis=1)
        return X


class Log1pColumns(BaseEstimator, TransformerMixin):
    """log1p on selected non-negative columns (negatives clipped to 0 as a guard)."""

    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = np.log1p(X[self.cols].clip(lower=0))
        return X


class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.cols)


def build_preprocessor(use_pyth=False, cap=False, log_cols=(), drop_cols=(), scale=True):
    """Leakage-safe preprocessing pipeline.

    Order: impute -> (Pythagorean) -> (IQR cap) -> (log1p) -> (drop) ->
           StandardScaler on numerics + OneHotEncoder(handle_unknown="ignore") on Team Name.
    Every step is fitted on the training fold only when used inside a Pipeline.
    """
    imputer = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), NUM),
         ("cat", SimpleImputer(strategy="most_frequent"), CAT)],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    steps = [("impute", imputer)]
    if use_pyth:
        steps.append(("pyth", PythagoreanExpectation()))
    if cap:
        steps.append(("cap", IQRCapper([c for c in NUM if c != "Year"])))
    if log_cols:
        steps.append(("log", Log1pColumns(list(log_cols))))
    if drop_cols:
        steps.append(("drop", DropColumns(list(drop_cols))))

    num_cols = [c for c in NUM + (["Pythagorean"] if use_pyth else []) if c not in drop_cols]
    final = ColumnTransformer(
        [("num", StandardScaler() if scale else "passthrough", num_cols),
         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)]
    ).set_output(transform="pandas")
    steps.append(("encode_scale", final))
    return Pipeline(steps)


def make_pipeline(estimator, **prep_kwargs):
    return Pipeline([("prep", build_preprocessor(**prep_kwargs)), ("model", estimator)])


def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": float(np.sqrt(mse)), "R2": r2_score(y_true, y_pred)}


def classification_metrics(y_true, y_pred, score):
    """`score` = predicted probability (or decision score) of the positive class."""
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, score)}


def md_table(df, floatfmt=".3f"):
    """Minimal DataFrame -> Markdown table (no extra dependency)."""
    df = df.reset_index() if df.index.name or not isinstance(df.index, pd.RangeIndex) else df
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        cells = [format(v, floatfmt) if isinstance(v, (float, np.floating)) else str(v) for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


1. Web Scraping

---



In [2]:
import datetime as dt, hashlib, json, shutil, time
import requests
from bs4 import BeautifulSoup
import datetime as dt
import hashlib
import json
import shutil
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup


BASE_URL = "https://www.scrapethissite.com/pages/forms/"
PER_PAGE = 100
HEADERS = {"User-Agent": "Mozilla/5.0 (university project - polite scraper)"}

# CSS class of each <td> -> output column name
CLASS_TO_COL = {"name": "Team Name", "year": "Year", "wins": "Wins", "losses": "Losses",
                "ot-losses": "OT Losses", "pct": "Win %", "gf": "GF", "ga": "GA", "diff": "+/-"}
COLUMNS = list(CLASS_TO_COL.values())


def parse_page(html):
    """Return (list of row dicts, number of pages found in the pagination bar)."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.select("tr.team"):
        tds = tr.find_all("td")
        rec = {}
        for td in tds:
            for cls, col in CLASS_TO_COL.items():
                if cls in (td.get("class") or []):
                    rec[col] = td.get_text(strip=True)
        if len(rec) != len(COLUMNS):  # fallback: positional order
            cells = [td.get_text(strip=True) for td in tds]
            if len(cells) != len(COLUMNS):
                continue
            rec = dict(zip(COLUMNS, cells))
        rows.append(rec)
    pages = [int(a.get_text(strip=True)) for a in soup.select("ul.pagination a")
             if a.get_text(strip=True).isdigit()]
    return rows, (max(pages) if pages else 1)


def fetch(page):
    r = requests.get(BASE_URL, params={"page_num": page, "per_page": PER_PAGE},
                     headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text


def main():
    all_rows, page, n_pages = [], 1, 1
    while page <= n_pages:
        rows, n_pages = parse_page(fetch(page))
        print(f"page {page}/{n_pages}: {len(rows)} rows")
        if not rows:
            break
        all_rows += rows
        page += 1
        time.sleep(1)  # be polite

    df = pd.DataFrame(all_rows, columns=COLUMNS)
    # Convert types only (no value is changed). Blank cells become NaN.
    df["Team Name"] = df["Team Name"].str.strip()
    for c in COLUMNS[1:]:
        df[c] = pd.to_numeric(df[c].replace("", pd.NA), errors="coerce")
    df["Year"] = df["Year"].astype(int)

    df.to_csv(RAW_CSV, index=False)
    shutil.copyfile(RAW_CSV, RAW_BACKUP)  # untouched backup
    sha = hashlib.sha256(RAW_CSV.read_bytes()).hexdigest()
    meta = {"source_url": BASE_URL, "per_page": PER_PAGE,
            "collected_on_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "rows": int(len(df)), "columns": COLUMNS, "sha256_raw_csv": sha}
    RAW_META.write_text(json.dumps(meta, indent=2))
    print(f"Saved {len(df)} rows -> {RAW_CSV}\nMetadata -> {RAW_META}")

import datetime as dt, hashlib, json, shutil, time
import requests
from bs4 import BeautifulSoup
import datetime as dt
import hashlib
import json
import shutil
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup


BASE_URL = "https://www.scrapethissite.com/pages/forms/"
PER_PAGE = 100
HEADERS = {"User-Agent": "Mozilla/5.0 (university project - polite scraper)"}

# CSS class of each <td> -> output column name
CLASS_TO_COL = {"name": "Team Name", "year": "Year", "wins": "Wins", "losses": "Losses",
                "ot-losses": "OT Losses", "pct": "Win %", "gf": "GF", "ga": "GA", "diff": "+/-"}
COLUMNS = list(CLASS_TO_COL.values())
OUT_COLUMNS = COLUMNS + ["collected_at"]


def now_iso():
    """Local date-time with UTC offset, e.g. 2026-11-27T11:30:05+05:00."""
    return dt.datetime.now().astimezone().isoformat(timespec="seconds")


def parse_page(html):
    """Return (list of row dicts, number of pages found in the pagination bar)."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.select("tr.team"):
        tds = tr.find_all("td")
        rec = {}
        for td in tds:
            for cls, col in CLASS_TO_COL.items():
                if cls in (td.get("class") or []):
                    rec[col] = td.get_text(strip=True)
        if len(rec) != len(COLUMNS):  # fallback: positional order
            cells = [td.get_text(strip=True) for td in tds]
            if len(cells) != len(COLUMNS):
                continue
            rec = dict(zip(COLUMNS, cells))
        rows.append(rec)
    pages = [int(a.get_text(strip=True)) for a in soup.select("ul.pagination a")
             if a.get_text(strip=True).isdigit()]
    return rows, (max(pages) if pages else 1)


def fetch(page):
    r = requests.get(BASE_URL, params={"page_num": page, "per_page": PER_PAGE},
                     headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text


def main():
    start = now_iso()
    all_rows, page, n_pages, n_requests = [], 1, 1, 0
    while page <= n_pages:
        html = fetch(page)
        n_requests += 1
        stamp = now_iso()                       # moment this page was collected
        rows, n_pages = parse_page(html)
        print(f"page {page}/{n_pages}: {len(rows)} rows (collected_at {stamp})")
        if not rows:
            break
        for r in rows:
            r["collected_at"] = stamp           # added to every row while collecting
        all_rows += rows
        page += 1
        time.sleep(1)  # be polite
    end = now_iso()

    df = pd.DataFrame(all_rows, columns=OUT_COLUMNS)
    # Convert types only (no value is changed). Blank cells become NaN.
    df["Team Name"] = df["Team Name"].str.strip()
    for c in COLUMNS[1:]:
        df[c] = pd.to_numeric(df[c].replace("", pd.NA), errors="coerce")
    df["Year"] = df["Year"].astype(int)

    df.to_csv(RAW_CSV, index=False)
    shutil.copyfile(RAW_CSV, RAW_BACKUP)  # untouched backup
    sha = hashlib.sha256(RAW_CSV.read_bytes()).hexdigest()
    meta = {"team_roll_numbers": TEAM_ROLL_NUMBERS, "source_url": BASE_URL, "per_page": PER_PAGE,
            "collection_start": start, "collection_end": end,
            "requests": n_requests, "pages": page - 1,
            "rows": int(len(df)), "columns": OUT_COLUMNS, "sha256_raw_csv": sha}
    RAW_META.write_text(json.dumps(meta, indent=2))
    print(f"Saved {len(df)} rows -> {RAW_CSV}\nMetadata -> {RAW_META}")




if RAW_CSV.exists():
    print("Raw CSV already exists, skipping scrape:", RAW_CSV)
else:
    main()


if RAW_CSV.exists():
    print("Raw CSV already exists, skipping scrape:", RAW_CSV)
else:
    main()

page 1/6: 100 rows (collected_at 2026-11-27T07:32:27+00:00)
page 2/6: 100 rows (collected_at 2026-11-27T07:32:29+00:00)
page 3/6: 100 rows (collected_at 2026-11-27T07:32:30+00:00)
page 4/6: 100 rows (collected_at 2026-11-27T07:32:31+00:00)
page 5/6: 100 rows (collected_at 2026-11-27T07:32:33+00:00)
page 6/6: 82 rows (collected_at 2026-11-27T07:32:34+00:00)
Saved 582 rows -> /content/data/hockey_raw.csv
Metadata -> /content/data/hockey_raw_metadata.json
Raw CSV already exists, skipping scrape: /content/data/hockey_raw.csv


Data collection log
Printed from the metadata written at scrape time; the SHA-256 is recomputed from the raw file so it can be matched to the submitted data/hockey_raw.csv

In [3]:
meta = json.loads(RAW_META.read_text())
raw_df = pd.read_csv(RAW_CSV)
sha_now = hashlib.sha256(RAW_CSV.read_bytes()).hexdigest()
print("=" * 78); print("DATA COLLECTION LOG"); print("=" * 78)
print("Team roll numbers      :", ", ".join(meta["team_roll_numbers"]))
print("Source URL             :", meta["source_url"], f"(per_page={meta['per_page']})")
print("Collection started     :", meta["collection_start"])
print("Collection finished    :", meta["collection_end"])
print("HTTP requests / pages  :", meta["requests"], "/", meta["pages"])
print("Rows collected         :", meta["rows"], "(rows in file now:", len(raw_df), ")")
print("collected_at column    :", "collected_at" in raw_df.columns,
      "| missing values in it:", int(raw_df["collected_at"].isna().sum()),
      "| first:", raw_df["collected_at"].min(), "| last:", raw_df["collected_at"].max())
print("SHA-256 (recorded)     :", meta["sha256_raw_csv"])
print("SHA-256 (file now)     :", sha_now)
print("Fingerprint match      :", sha_now == meta["sha256_raw_csv"])
print("Raw file               :", RAW_CSV.name)
display(raw_df.head(3))


DATA COLLECTION LOG
Team roll numbers      : 25L-2558
Source URL             : https://www.scrapethissite.com/pages/forms/ (per_page=100)
Collection started     : 2026-11-27T07:32:27+00:00
Collection finished    : 2026-11-27T07:32:36+00:00
HTTP requests / pages  : 6 / 6
Rows collected         : 582 (rows in file now: 582 )
collected_at column    : True | missing values in it: 0 | first: 2026-11-27T07:32:27+00:00 | last: 2026-11-27T07:32:34+00:00
SHA-256 (recorded)     : 98e30be2fa1a3232b0e1fd6bd114d6c04718700c3a8753a18676082debb91841
SHA-256 (file now)     : 98e30be2fa1a3232b0e1fd6bd114d6c04718700c3a8753a18676082debb91841
Fingerprint match      : True
Raw file               : hockey_raw.csv


,Team Name,Year,Wins,Losses,OT Losses,Win %,GF,GA,+/-,collected_at
0,Boston Bruins,1990,44,24,NaN,0.550,299,264,35,2026-11-27T07:32:27+00:00
1,Buffalo Sabres,1990,31,30,NaN,0.388,292,278,14,2026-11-27T07:32:27+00:00
2,Calgary Flames,1990,46,26,NaN,0.575,344,263,81,2026-11-27T07:32:27+00:00
